In [1]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import yfinance as yf

In [2]:
stocks_name = ['BTC-USD', 'ETH-USD', 'SPY', 'GLD', '^VIX']

In [3]:
stocks_data = yf.download(stocks_name, start='2018-01-01', end=pd.Timestamp.now().date(), interval='1wk') #interval='5d')

[*********************100%***********************]  5 of 5 completed


In [4]:
df_stocks = pd.DataFrame(stocks_data).dropna()

In [5]:
df_stocks_close = df_stocks['Close'].dropna().drop(columns=['^VIX']).copy()

Normalized BTC/USD, SPY and GLD

In [6]:
normalized_df = (df_stocks_close / df_stocks_close.iloc[0])

In [20]:
fig = make_subplots(
    rows=1,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[1.5]
)

fig.add_trace(go.Scatter(
    x=normalized_df.index,
    y=normalized_df['BTC-USD']
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=normalized_df.index,
    y=normalized_df['SPY']
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=normalized_df.index,
    y=normalized_df['GLD']
), row=1, col=1)

fig.update_layout(
    title='Normalized Prices (BTC, SPY, GLD)',
    xaxis_title='Date',
    yaxis_title='Normalized Price',
    height=600,
    showlegend=True,
    template='plotly_dark'
)

fig.show()

Copy the DataFrame to a new variable

In [8]:
df_stocks_close_criptos = df_stocks['Close'].copy()

Thechnical Indicators

In [9]:
###
# Calculate SMAs for BTC-USD
df_stocks_close_criptos['BTC-USD_EMA-50'] = df_stocks_close_criptos['BTC-USD'].ewm(span=50).mean()
df_stocks_close_criptos['BTC-USD_EMA-100'] = df_stocks_close_criptos['BTC-USD'].ewm(span=100).mean()
df_stocks_close_criptos['BTC-USD_EMA-200'] = df_stocks_close_criptos['BTC-USD'].ewm(span=200).mean()
###
# Calculate SMAs for ETH-USD
df_stocks_close_criptos['ETH-USD_EMA-50'] = df_stocks_close_criptos['ETH-USD'].ewm(span=50).mean()
df_stocks_close_criptos['ETH-USD_EMA-100'] = df_stocks_close_criptos['ETH-USD'].ewm(span=100).mean()
df_stocks_close_criptos['ETH-USD_EMA-200'] = df_stocks_close_criptos['ETH-USD'].ewm(span=200).mean()
###

MACD

In [10]:
###
#MACD
def calculate_macd(df, fast=12, slow=26, signal=9):
    exp1 = df_stocks['Close']['BTC-USD'].ewm(span=fast).mean()
    exp2 = df_stocks['Close']['BTC-USD'].ewm(span=slow).mean()
    macd = exp1 - exp2
    signal = macd.ewm(span=signal).mean()
    histogram = macd - signal
    return macd, signal, histogram


macd, signal, histogram = calculate_macd(df_stocks)

BB %B - Bollinger Bands Percentage Bandwidth

In [11]:
window = 20
std_dev = 2

price = df_stocks["Close"]["BTC-USD"]
middle = price.rolling(window).mean()
std = price.rolling(window).std()
upper = middle + std_dev * std
lower = middle - std_dev * std

bb_percent_b = (price - lower) / (upper - lower)

Stochastic + RSI

In [12]:
# RSI
def calculate_rsi(prices, window=14):
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# Stochastic Oscillator
def calculate_stochastic(high, low, close, k_window=14, d_window=3):
    lowest_low = low.rolling(window=k_window).min()
    highest_high = high.rolling(window=k_window).max()
    k_percent = 100 * ((close - lowest_low) / (highest_high - lowest_low))
    d_percent = k_percent.rolling(window=d_window).mean()
    return k_percent, d_percent

# Calcular para BTC-USD
rsi = calculate_rsi(df_stocks['Close']['BTC-USD'])
stoch_k, stoch_d = calculate_stochastic(
    df_stocks['High']['BTC-USD'],
    df_stocks['Low']['BTC-USD'],
    df_stocks['Close']['BTC-USD']
)

Returns of the investments in multiplication

In [13]:
stocks_returns_week_df = (df_stocks_close.pct_change().dropna())#Day variation
acumulated_returns = (1 + stocks_returns_week_df).cumprod()

Log Return

In [14]:
stocks_returns_log_df = np.log(df_stocks_close/ df_stocks_close.shift(1)).dropna()

Volatility Weeks Sharpe & Sortino

In [15]:
trading_weeks_sharpe = 60 #30
trading_weeks_sortino = 90
volatility_weeks_sharpe = stocks_returns_log_df.rolling(window=trading_weeks_sharpe).std()*np.sqrt(trading_weeks_sharpe)
volatility_weeks_sortino = stocks_returns_log_df.rolling(window=trading_weeks_sortino).std()*np.sqrt(trading_weeks_sortino)

Sharpe Ratio Weeks(52)

In [16]:
rf = 0.01/52
sharpe_ratio = (stocks_returns_log_df.rolling(window=trading_weeks_sharpe).mean() - rf)*trading_weeks_sharpe/volatility_weeks_sharpe

Sortino Ratio

In [17]:
sortino_vol = stocks_returns_log_df[stocks_returns_log_df<0].rolling(window=trading_weeks_sortino, center=True, min_periods=10).std()*np.sqrt(trading_weeks_sortino)
sortino_ratio = (stocks_returns_log_df.rolling(window=trading_weeks_sortino).mean() - rf)*trading_weeks_sortino/sortino_vol

Data Conditioning of BTC and SPY

In [18]:
# %% [markdown]
# ## Data Conditioning BTC

nv_shp_btc_top = 2.2
nv_srt_btc_top = 3.5
nv_shp_btc_bottom = -1.5
nv_srt_btc_bottom = -2.0
mask_btc_sortino_top = sortino_ratio['BTC-USD'] >= 3.7
mask_btc_sharpe_top = sharpe_ratio['BTC-USD'] > nv_shp_btc_top
mask_btc_sharpe_sortino_top = ((sortino_ratio['BTC-USD'] >= nv_srt_btc_top) & mask_btc_sharpe_top)
mask_btc_sortino_reversion_down = (sharpe_ratio['BTC-USD'] < 1.2) & (sortino_ratio['BTC-USD'] >= nv_srt_btc_top) | ((sharpe_ratio['BTC-USD'] > nv_shp_btc_top) & (sortino_ratio['BTC-USD'] >= nv_srt_btc_top))
mask_btc_sharpe_bottom = sharpe_ratio['BTC-USD'] < nv_shp_btc_bottom
mask_btc_sortino_bottom = (sortino_ratio['BTC-USD'] < nv_srt_btc_bottom) & mask_btc_sharpe_bottom
mask_btc_sharpe_sortino_bottom = ((sortino_ratio['BTC-USD'] < nv_srt_btc_bottom) & mask_btc_sharpe_bottom)
mask_btc_sortino_reversion_up = (sharpe_ratio['BTC-USD'] > 0) & (sortino_ratio['BTC-USD'] <= -1.5)

# %% [markdown]
# ## Data Conditioning SPY
# %%

nv_shp_spy_top = 2.5
nv_srt_spy_top = 3.2
nv_shp_spy_bottom = 0
nv_srt_spy_bottom = 0
mask_spy_sortino_top = sortino_ratio['SPY'] >= nv_srt_spy_top
mask_spy_sharpe_top = sharpe_ratio['SPY'] > nv_shp_spy_top
mask_spy_sharpe_sortino_top = ((sortino_ratio['SPY'] >= nv_srt_spy_top) & mask_spy_sharpe_top)
mask_spy_sortino_reversion_down = (sharpe_ratio['SPY'] < nv_shp_spy_bottom) & (sortino_ratio['SPY'] >= 3.0)
mask_spy_sharpe_bottom = sharpe_ratio['SPY'] < -0.6
mask_spy_sortino_bottom = (sortino_ratio['SPY'] < nv_srt_spy_bottom)
mask_spy_sharpe_sortino_bottom = ((sortino_ratio['SPY'] < 0) & mask_spy_sharpe_bottom)
mask_spy_sortino_reversion_up = (sharpe_ratio['SPY'] > 1.0) & (sortino_ratio['SPY'] <= 0)


Graphic

In [21]:
# %% [markdown]
# ## Data Visualization
# %%

fig = make_subplots(
    rows=7,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.005,
    row_heights=[2.0, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5]
)

# %% [markdown]
# ## Data Plotting Price BTC
# %%

fig.add_trace(go.Candlestick(
    x=df_stocks.index,
    open=df_stocks['Open']['BTC-USD'],
    high=df_stocks['High']['BTC-USD'],
    low=df_stocks['Low']['BTC-USD'],
    close=df_stocks['Close']['BTC-USD'],
    name='BTC-USD',

), row=1, col=1)

###
# Plot BTC SMAs
fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_EMA-50'],
    name='EMA-50',
    line=dict(color='lightgreen')
), row=1, col=1)


fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_EMA-100'],
    name='EMA-100',
    line=dict(color='lightblue')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_EMA-200'],
    name='EMA-200',
    line=dict(color='white')
), row=1, col=1)

###
#Plot MACD
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=macd,
    name='MACD',
    marker_color='lightblue',
    opacity=0.7
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=signal,
    name='Signal',
    marker_color='lightcoral',
    opacity=0.7
    ), row=2, col=1)

fig.add_trace(go.Bar(
    x=df_stocks.index,
    y=histogram,
    name='Histogram',
    marker_color=histogram,
    ), row=2, col=1)

# %% [markdown]
# ## Data Plotting Sharpe and Sortino Ratios BTC
# %%

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index,
    y=sharpe_ratio['BTC-USD'],
    line=dict(color='white'),
    name='Sharpe Ratio BTC',
), row=3, col=1)

fig.add_trace(go.Bar(
    x=sortino_ratio.index,
    y=sortino_ratio['BTC-USD'],
    marker_color=sortino_ratio['BTC-USD'],
    name='Sortino Ratio BTC',
), row=3, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sharpe_sortino_top],
    y=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_top],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_top])*4,
                color=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_top],
                colorscale='YlOrRd',
                symbol='diamond'),
    name='Sortino BTC > 3.5 & Sharpe > 2.0'),
    row=3, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sharpe_sortino_bottom],
    y=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_bottom])*6,
                color=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_bottom],
                colorscale='BuGn',
                symbol='diamond'),
    name='Sort < -2.0 & Shar < -2.0'),
    row=3, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_btc_sharpe_top],
    y=sharpe_ratio['BTC-USD'][mask_btc_sharpe_top],
    mode='markers',
    marker=dict(size=sharpe_ratio['BTC-USD'][mask_btc_sharpe_top]*3,
                color='lightcoral',
                symbol='circle'),
    name='Sharpe BTC > 2'),
    row=3, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_top],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_top],
    mode='markers',
    marker=dict(size=sortino_ratio['BTC-USD'][mask_btc_sortino_top]*2,
                color=sortino_ratio['BTC-USD'][mask_btc_sortino_top],
                colorscale='PuRd',
                symbol='circle'),
    name='Sortino BTC > 3.5'),
    row=3, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_btc_sharpe_bottom],
    y=sharpe_ratio['BTC-USD'][mask_btc_sharpe_bottom],
    mode='markers',
    marker=dict(size=abs(sharpe_ratio['BTC-USD'][mask_btc_sharpe_bottom])*3,
                color='lightgreen',
                symbol='circle'),
    name='Sharpe BTC < -2'),
    row=3, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_reversion_down],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_down],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_down])*2,
                color=sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_down],
                colorscale='YlOrRd',
                symbol='circle'),
    name=f'Sort > 3.5 & Shar < 0'),
    row=3, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_bottom],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_bottom])*3,
                color='green',
                symbol='circle'),
    name=f'Sortino BTC < -2.0'),
    row=3, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_reversion_up],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_up],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_up])*5,
                color='lightblue',
                symbol='star'),
    name='Sort < -1.5 & Shar > 0'),
    row=3, col=1
)

# %% [markdown]
# ## Data Plotting Sharpe and Sortino Ratios SPY
# %%

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index,
    y=sharpe_ratio['SPY'],
    line=dict(color='white'),
    name='Sharpe Ratio SPY',
), row=4, col=1)

fig.add_trace(go.Bar(
    x=sortino_ratio.index,
    y=sortino_ratio['SPY'],
    marker_color=sortino_ratio['SPY'],
    name='Sortino Ratio SPY',
), row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sharpe_sortino_top],
    y=sortino_ratio['SPY'][mask_spy_sharpe_sortino_top],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sharpe_sortino_top])*3,
                color=sortino_ratio['SPY'][mask_spy_sharpe_sortino_top],
                colorscale='YlOrRd',
                symbol='diamond'),
    name='Sortino SPY > 3.2 & Sharpe > 2.5'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sharpe_sortino_bottom],
    y=sortino_ratio['SPY'][mask_spy_sharpe_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sharpe_sortino_bottom])*35,
                color=sortino_ratio['SPY'][mask_spy_sharpe_sortino_bottom],
                colorscale='BuGn',
                symbol='diamond'),
    name='Sort < 0 & Shar < -0.2'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_spy_sharpe_top],
    y=sharpe_ratio['SPY'][mask_spy_sharpe_top],
    mode='markers',
    marker=dict(size=sharpe_ratio['SPY'][mask_spy_sharpe_top]*3,
                color='lightcoral',
                symbol='circle'),
    name='Sharpe SPY > 2'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_top],
    y=sortino_ratio['SPY'][mask_spy_sortino_top],
    mode='markers',
    marker=dict(size=sortino_ratio['SPY'][mask_spy_sortino_top]*2,
                color=sortino_ratio['SPY'][mask_spy_sortino_top],
                colorscale='YlOrRd',
                symbol='circle'),
    name='Sortino SPY > 3.2'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_spy_sharpe_bottom],
    y=sharpe_ratio['SPY'][mask_spy_sharpe_bottom],
    mode='markers',
    marker=dict(size=abs(sharpe_ratio['SPY'][mask_spy_sharpe_bottom])*8,
                color='lightgreen',
                symbol='circle'),
    name='Sharpe SPY < -0.2'),
    row=4, col=1
)


fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_reversion_down],
    y=sortino_ratio['SPY'][mask_spy_sortino_reversion_down],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sortino_reversion_down])*2,
                color='red',
                symbol='circle'),
    name=f'Sort > 3.5'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_bottom],
    y=sortino_ratio['SPY'][mask_spy_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sortino_bottom])*12,
                color='green',
                symbol='circle'),
    name=f'Sortino SPY < 0'),
    row=4, col=1
)

'''fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_reversion_up],
    y=sortino_ratio['SPY'][mask_spy_sortino_reversion_up],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sortino_reversion_up])*20,
                color='lightblue',
                symbol='star'),
    name='Sort < -1.0 & Shar > 1.0'),
    row=4, col=1
)


# %% [markdown]
# ## Data Plotting Sharpe and Sortino Ratios GLD
# %%

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index,
    y=sharpe_ratio['GLD'],
    line=dict(color='white'),
    name='Sharpe Ratio GLD',
), row=5, col=1)

fig.add_trace(go.Bar(
    x=sortino_ratio.index,
    y=sortino_ratio['GLD'],
    marker_color=sortino_ratio['GLD'],
    name='Sortino Ratio GLD',
), row=5, col=1)'''


# %% [markdown]
# ## Data Plotting Bollinger Bands %B
# %%

fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=bb_percent_b,
    name='BB %B',
    line=dict(color='#ff0000', width=1),
    opacity=0.7,
), row=5, col=1)

'''fig.update_layout(
    yaxis3=dict(
        autorange=False,
        range=[-1, 1.5],
        title="BB %B"
    )
)'''

fig.add_hline(y=0, line_dash="dot", line_color="lightgreen", row=5, col=1)
fig.add_hline(y=0.50, line_dash="dot", line_color="gray", row=5, col=1)
fig.add_hline(y=1, line_dash="dot", line_color="lightcoral", row=5, col=1)

# %% [markdown]
# ## Data Plotting RSI and Stochastic
# %%

fig.add_trace(go.Scatter(
            x=df_stocks.index,
            y=rsi,
            name='RSI',
            line=dict(color='purple', width=1)
), row=6, col=1)

fig.add_trace(go.Scatter(
            x=df_stocks.index,
            y=stoch_d,
            name='%D',
            line=dict(color='red', width=1)
), row=6, col=1)

fig.add_hline(y=75, line_dash='dash', line_color='red', row=6, col=1)
fig.add_hline(y=25, line_dash='dash', line_color='red', row=6, col=1)

# %% [markdown]
# ## Data Plotting VIX
# %%

fig.add_trace(go.Bar(
    x=df_stocks.index,
    y=df_stocks['Close']['^VIX'],
    name='VIX',
    marker_color=df_stocks['Close']['^VIX']
), row=7, col=1)

'''fig.add_trace(go.Scatter(
    x=volatility_weeks_sharpe.index,
    y=volatility_weeks_sharpe['BTC-USD'],
    line=dict(color='purple'),
    name='Volatily BTC-USD',
), row=6, col=1)

fig.add_trace(go.Scatter(
    x=volatility_weeks_sharpe.index,
    y=volatility_weeks_sharpe['SPY'],
    line=dict(color='lightgreen'),
    name='Volatily SPY',
), row=6, col=1)

fig.add_trace(go.Scatter(
    x=volatility_weeks_sharpe.index,
    y=volatility_weeks_sharpe['GLD'],
    line=dict(color='gold'),
    name='Volatility GLD',
), row=6, col=1)'''

# %% [markdown]
# ## Data Plotting End
# %%

fig.update_layout(
    title= f'Weekly Analysis - {df_stocks.index[0].strftime("%d %b %Y")} to {df_stocks.index[-1].strftime("%d %b %Y")}',
    title_x=0.5,
    showlegend=False,
    height=1500,
    width=1050,
    yaxis_title='Price',
    yaxis2_title='MACD',
    yaxis3_title='SHP&STO Ratios BTC',
    yaxis4_title='SHP&STO Ratios SPY',
    yaxis5_title='BB %B',
    yaxis6_title='RSI & Stochastic',
    yaxis7_title='VIX',
    template='plotly_dark',
    xaxis_rangeslider_visible=False,
)
fig.show()